# 02 — Phân tích Random Forest (mô hình chính)

Notebook này **hiển thị** kết quả của `python -m src.rf_analysis` (không tính lại), để mọi số liệu
khớp với `reports/results_snapshot.md` §9. Nếu chưa có kết quả, ô bên dưới sẽ tự chạy bản nhanh.

Nội dung:
1. Validation curve theo `n_estimators` và `max_depth` — tham số đã tune nằm ở đâu trên đường bias–variance.
2. OOB vs cross-validation vs test.
3. Độ quan trọng đặc trưng theo 3 cách: MDI, permutation, SHAP — và mức đồng thuận giữa chúng.
4. Giải thích cục bộ (SHAP) cho 3 khách hàng: TP, FN, FP.

**Đổi `DATASET`** sang dataset chính sau khi chốt D1 (`'ibm'` hoặc `'iranian'`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display

DATASET = 'ibm'   # 'ibm' | 'iranian' | 'orange'
RESULTS = ROOT / 'results' / 'rf_analysis'
FIGURES = ROOT / 'figures' / 'rf'

if not (RESULTS / f'{DATASET}_importance.csv').exists():
    import os
    os.chdir(ROOT)
    from src.rf_analysis import analyse_dataset, OUT_DIR, FIG_DIR
    OUT_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)
    analyse_dataset(DATASET, quick=True)

pd.set_option('display.max_colwidth', 80)

## 1. Validation curve

Trục tung là average precision (AP) trên cross-validation của **tập train**. Khoảng cách giữa đường
Train và đường CV thể hiện mức overfit; đường CV đi ngang nghĩa là tăng tham số không còn lợi.
Đường nét đứt là giá trị đã chọn khi tune.

In [ ]:
display(pd.read_csv(RESULTS / f'{DATASET}_validation_curve.csv'))
display(Image(filename=str(FIGURES / f'validation_curve_{DATASET}.png')))

## 2. OOB vs CV vs test

OOB (out-of-bag) là ước lượng "miễn phí" của RF: mỗi cây được đánh giá trên các dòng không nằm
trong mẫu bootstrap của nó. Nếu OOB ≈ CV ≈ test thì đánh giá đáng tin. Với Iranian, OOB có thể
lạc quan vì một bản trùng có thể nằm trong mẫu bootstrap trong khi bản kia là OOB.

In [ ]:
summary = pd.read_csv(RESULTS / 'rf_summary.csv')
display(summary[summary['dataset'] == DATASET].T)

## 3. Độ quan trọng đặc trưng — 3 cách

- **MDI**: tổng mức giảm impurity; nhanh nhưng thiên vị biến có nhiều giá trị và tính trên tập train.
- **Permutation**: AP trên test giảm bao nhiêu khi xáo trộn một cột gốc; phản ánh đúng hiệu năng,
  nhưng chia điểm giữa các biến tương quan (có thể âm).
- **SHAP**: đóng góp trung bình |SHAP| của từng biến vào xác suất dự đoán.

Hệ số Spearman (ρ) trong bảng ở mục 2 cho biết các cách xếp hạng có đồng thuận không.

In [ ]:
importance = pd.read_csv(RESULTS / f'{DATASET}_importance.csv')
display(importance.head(12))
display(Image(filename=str(FIGURES / f'importance_methods_{DATASET}.png')))

## 4. Giải thích cục bộ — vì sao mô hình đúng/sai với từng khách

Thanh đỏ đẩy xác suất churn **lên**, thanh xanh kéo **xuống**. FN (bỏ sót) và FP (báo nhầm) là
chất liệu tốt cho phần thảo luận hạn chế của mô hình. Đây là tương quan mà mô hình học được,
**không phải quan hệ nhân quả**.

In [ ]:
local = pd.read_csv(RESULTS / f'{DATASET}_local_shap.csv')
display(local[['case', 'churn_probability', 'actual_churn', 'rank', 'feature', 'value', 'contribution']])
display(Image(filename=str(FIGURES / f'local_shap_{DATASET}.png')))

## 5. Calibration (bổ sung)

Reliability diagram của các cách calibrate (từ `python -m src.calibration_analysis`), dùng cho
mục 5.2 và cho Nhật Linh ở mục 5.3.

In [ ]:
path = FIGURES / f'reliability_{DATASET}.png'
if path.exists():
    display(Image(filename=str(path)))
    display(pd.read_csv(ROOT / 'results' / 'calibration' / 'calibration_comparison.csv'))
else:
    print('Chưa có: chạy python -m src.calibration_analysis --dataset', DATASET)